# SmartProduce — Train Produce Classifier

Trains a MobileNetV2 image classifier on Fruits-360 images, then exports to TF.js so it runs in the browser.

**Runtime → Change runtime type → T4 GPU** before running.

At the end you'll download a `tfjs_model.zip` — hand that to Claude Code to wire into the app.

In [ ]:
# ── 1. Install converter ────────────────────────────────────────────────────
!pip install tensorflowjs -q

In [ ]:
import tensorflow as tf
import tensorflowjs as tfjs
print('TF', tf.__version__, '| TFJS', tfjs.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── 2. Download images from Fruits-360 ─────────────────────────────────────
import os, urllib.request, urllib.parse, json, time

CLASSES = {
    'banana':      ['Banana', 'Banana Lady Finger'],
    'tomato':      ['Tomato 1', 'Tomato 2', 'Tomato 3'],
    'apple':       ['Apple Red 1', 'Apple Red 2', 'Apple Granny Smith'],
    'mango':       ['Mango', 'Mango Red'],
    'carrot':      ['Carrot'],
    'potato':      ['Potato Red', 'Potato White'],
    'onion':       ['Onion Red', 'Onion White'],
    'coconut':     ['Cocos'],
    'watermelon':  ['Watermelon'],
    'pineapple':   ['Pineapple'],
    'grape':       ['Grape Blue', 'Grape White 2'],
    'lemon':       ['Lemon', 'Lemon Meyer'],
    'lime':        ['Lime'],
    'avocado':     ['Avocado'],
    'cucumber':    ['Cucumber Ripe'],
    'brinjal':     ['Eggplant'],
    'corn':        ['Corn'],
    'pomegranate': ['Pomegranate'],
    'ginger':      ['Ginger Root'],
    'garlic':      ['Garlic'],
    'pepper':      ['Pepper Red', 'Pepper Green'],
    'papaya':      ['Papaya'],
    'pear':        ['Pear'],
    'peach':       ['Peach'],
}

IMAGES_PER_CLASS = 30
BASE_DIR = '/content/images'

def list_github_folder(folder):
    url = f"https://api.github.com/repos/Horea94/Fruit-Images-Dataset/contents/Training/{urllib.parse.quote(folder)}"
    req = urllib.request.Request(url, headers={'User-Agent': 'smartproduce'})
    try:
        with urllib.request.urlopen(req, timeout=15) as r:
            return json.loads(r.read())
    except Exception as e:
        print(f'  ⚠ {folder}: {e}')
        return []

for label, folders in CLASSES.items():
    dest = f'{BASE_DIR}/{label}'
    os.makedirs(dest, exist_ok=True)
    collected = 0
    for folder in folders:
        if collected >= IMAGES_PER_CLASS:
            break
        files = list_github_folder(folder)
        for f in files:
            if collected >= IMAGES_PER_CLASS:
                break
            if not f['name'].endswith('.jpg'):
                continue
            try:
                urllib.request.urlretrieve(f['download_url'], f"{dest}/{label}_{collected:03d}.jpg")
                collected += 1
            except:
                pass
            time.sleep(0.03)
    print(f'  {label}: {collected} images')

print('\nDownload complete.')

In [ ]:
# ── 3. Build data pipeline ──────────────────────────────────────────────────
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH    = 16

# Only keep classes that have enough images
classes = sorted([
    d for d in os.listdir(BASE_DIR)
    if os.path.isdir(f'{BASE_DIR}/{d}') and len(os.listdir(f'{BASE_DIR}/{d}')) >= 8
])
print(f'{len(classes)} classes: {classes}')

gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.25,
    horizontal_flip=True,
    brightness_range=[0.75, 1.25],
    validation_split=0.2,
)

train_ds = gen.flow_from_directory(
    BASE_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
    class_mode='categorical', classes=classes, subset='training', seed=42
)
val_ds = gen.flow_from_directory(
    BASE_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
    class_mode='categorical', classes=classes, subset='validation', seed=42
)

print(f'Train batches: {len(train_ds)} | Val batches: {len(val_ds)}')

In [ ]:
# ── 4. Build model ──────────────────────────────────────────────────────────
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

n_classes = len(classes)

base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(n_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.summary()

In [ ]:
# ── 5. Phase 1 — train head only ────────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

h1 = model.fit(
    train_ds, validation_data=val_ds, epochs=12,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)
print(f'Best val acc: {max(h1.history["val_accuracy"]):.1%}')

In [ ]:
# ── 6. Phase 2 — fine-tune top layers ──────────────────────────────────────
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

h2 = model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)]
)
print(f'Best val acc: {max(h2.history["val_accuracy"]):.1%}')

In [ ]:
# ── 7. Export to TF.js ──────────────────────────────────────────────────────
import json

TFJS_DIR = '/content/tfjs_model'
tfjs.converters.save_keras_model(model, TFJS_DIR)

# Save class labels next to model
with open(f'{TFJS_DIR}/labels.json', 'w') as f:
    json.dump(classes, f)

print('Saved to', TFJS_DIR)
print('Classes:', classes)

In [ ]:
# ── 8. Download as zip ──────────────────────────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('/content/tfjs_model', 'zip', '/content/tfjs_model')
files.download('/content/tfjs_model.zip')
print('Download started — check your Downloads folder.')

## Done!

You now have `tfjs_model.zip`. Give it to Claude Code and say:

> "Here is the tfjs_model.zip — wire it into the SmartProduce app"

It will unzip it into `public/tfjs_model/`, install `@tensorflow/tfjs`, and swap the camera inference to use the local model instead of the API.